# Quick data inspection
## Create a dataframe


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "zip_code": [11777, 85278, 7086, 7086, 11777, 85278, 85361, 7086],
    "price_thousand_usd": [250, 320, 180, 450, 390, 275, 500, 210],
    "size_square_feet": [1800, 2200, 1200, 3000, 2500, 1900, 3500, 1400],
    "num_bedrooms": [3, 4, 2, 5, 4, 3, 6, 2],
    "property_type": ["House", "House", "Condo", "House",
                      "Townhouse", "Condo", "House", "Condo"],
    "condition_rating": ["Fair", "Good", "Good", "Excellent",
                         "Very Good", "Fair", "Excellent", "Good"],
    "stove_type": ["Gas", "Electric", "Electric", "Gas",
                   "Gas", "Electric", "Gas", "Electric"]
})

df

## Inspect type

We now have a preliminary understanding of the data:
- `zip_code`: seems numerical, but probably need to convert to categorical
- `price_thousand_usd`: numerical (continuous, aka, float), price in thousand dollars
- `size_squrre_feet`: numerical (continuous, aka, float), size in square feet
-  `num_bedrooms`: numerical (discrete, aka, integer), the number of bedrooms
- `property_type`: categorical (nominal)
- `condition_rating`: categorical (ordinal), from `Fair` to `Excellent`
-  `stove_type`: categorical (nominal)

In [ ]:
df.info()

## Inspect numerical variables

Units matter
- The unit of `price_thousand_usd' is thousand dollars, not dollars
- The unit for size is square feet, not square meters



In [ ]:
df.describe()

If you need to conduct calculation using dollars, you should create a new column to represent the price in dollars

In [ ]:
df['price_usd'] = df['price_thousand_usd'] * 1000

df

Notice:
- `zip_code` appears to be numerical, but behaves more like categorical.
- It is captured as integers, which can lead to the following problems:
    - it might by accident be transformed into float in calculation, which make no senses
    - zip codes can have 0 at the beginning, but integer cannot

A common solution is to:
- convert `zip_code` into a string, and
- add leading zeros to fill to five digits

This usually happens when you deal with unique IDs, e.g., GeoID for places, patient IDs, zip codes, FIPS (Federal Information Processing Standard) codes, etc. Therefore, it is important to understand what data you are looking at and process accordingly.


In [ ]:
df['zip_code'] = df['zip_code'].astype(str).str.zfill(5)


In [ ]:
df['zip_code']

## Inspect categorical variables

In [ ]:
df.info()

In [ ]:
categorical_cols = df.select_dtypes(exclude=['number']).columns

for col in categorical_cols:
    print(df[col].value_counts())
    print('\n')

### Convert nominal data to `category` dtype


In [ ]:
for nominal_data in ['zip_code', 'property_type', 'stove_type']:
    df[nominal_data] = df[nominal_data].astype('category')

In [ ]:
df.dtypes

### Convert ordinal data to ordered category dtype

Note: A categorical column is not equal to category dtype. It needs to be cast into a category dtype using `.astype('category')`

When casting ordinal data into category dtype, it is recommended to cast it into an ordered category dtype, to maintain the order. Two scenarios:
- If we are casting a column that is not yet category dtype, we can use `.astype()` to cast it into `pd.CategoricalDtype(categories=order_you_want_to_maintain, ordered=True)`.
- If you want to cast a column of category dtype to ordered category dtype, you can use one of the following two methods:
    - `.cat.set_categories(new_categories, ordered=True)`
    - `.cat.reorder_categories(new_categories, ordered=True)`

In [ ]:
# first inspect the unique values in condition_rating
df['condition_rating'].unique()

In [ ]:
condition_rating_order = ['Fair', 'Good', 'Very Good', 'Excellent']

df['condition_rating'] = df['condition_rating'].astype(pd.CategoricalDtype(categories=condition_rating_order, ordered=True))

df['condition_rating'].value_counts(sort=False) # set sort=False to maintain the order we set for condition_rating, instead of the order of count

#### Can we convert ordinal data into integers?

You absolutely can, although many times it is not considered the best practice.

However, in reality, many datasets are prepared in a way where ordinal data is recorded as integers. That said, the most important thing is to understand the data, understand what is appropriate way to deal with the data. Therefore, if you convert the `condition_rating` to integers, remember you can only rank them, but you cannot do arithmetic operation with those integers.

In [ ]:
# if we already have a ordered category dtype, we can simply use .cat.codes
df['condition_rating_numeric'] = df['condition_rating'].cat.codes

df[['condition_rating', 'condition_rating_numeric']]

In [ ]:
# in this case, the integer column's dtype is integer
df['condition_rating_numeric'].dtypes

In [ ]:
# you can also use .cat.rename_categories

df['condition_rating_numeric'] = df['condition_rating'].cat.rename_categories([0, 1, 2, 3])

df[['condition_rating','condition_rating_numeric']]

In [ ]:
# in this case, the type of this column is still category dtype
df['condition_rating_numeric'].dtypes

### Cross tabulation

In [ ]:
pd.crosstab(df['condition_rating'], df['stove_type'])

### Group inspect
If we want to inspect data within each group of a categorical variable, we can use `.groupby()`.


In [ ]:
df.groupby(by="property_type", observed=True)['price_usd'].mean()

In [ ]:
df.groupby(by='zip_code', observed=True)['size_square_feet'].describe()

In [ ]:
df.groupby(by='property_type', observed=True).agg(
    {'price_usd':['mean','sum'],
     'size_square_feet': ['mean','sum']}
)

You can even add custom functions to `.groupby()`

In [ ]:
df.groupby(by='property_type', observed=True).agg(
    mean_price_in_thousand_CNY = pd.NamedAgg(column='price_thousand_usd', aggfunc=lambda s: s.mean()*6.91),
    mean_size_in_square_meters = pd.NamedAgg(column='size_square_feet', aggfunc=lambda s: s.mean()/10.76),
)